Trabajo Práctico Especial: Fundamentos de la Ciencia de Datos (2026)

Grupo 14

Integrantes: Juan Cruz Higuera, Jeremías Aguilar, Santiago Silva Salvatierra

Ayudante: Duilio Deangeli

Dataset: Global Food & Nutrition Database 2026 – Health Scores & Allergens


Descripción General del Trabajo

El presente Trabajo Práctico Especial tiene como propósito aplicar los conceptos de la Ciencia de Datos sobre un conjunto de datos real vinculado a la composición nutricional, niveles de procesamiento y presencia de alérgenos en alimentos comerciales. A lo largo de esta notebook se desarrollará el flujo completo de trabajo: configuración del entorno, exploración inicial, limpieza y adecuación de variables, formulación y contraste de hipótesis estadísticas y modelado predictivo probabilístico.

Sección 1: Preparación del Entorno, Carga y Primer Diagnóstico de los Datos

Objetivo de la sección

Dejar listo el entorno de trabajo con las librerías necesarias, cargar el archivo de datos desde la carpeta local del proyecto y hacer una primera revisión completa de la tabla. La idea es conocer el tamaño de la muestra, mirar cómo vienen cargadas las filas, revisar qué tipos de datos asignó Pandas para detectar campos que necesiten corrección y analizar las pistas que nos permitan deducir de dónde salieron los datos y qué tipo de alimentos estamos estudiando realmente.

Tareas que vamos a realizar en esta sección:
Cargar las herramientas iniciales: Traer las librerías básicas para manipular tablas, calcular métricas y armar gráficos (1.1).

Leer el archivo: Cargar en memoria el dataset ubicado en la carpeta data/ usando una ruta relativa para que el proyecto funcione en cualquier máquina (1.2).   

Primer vistazo a los datos: Mostrar los primeros registros con head() para comprobar que las columnas se hayan separado bien y familiarizarnos con la información (1.3).

Revisar tamaño y tipos de datos: Usar shape e info() para ver la cantidad de productos y variables, anotando qué columnas numéricas vinieron guardadas como texto o con campos vacíos (1.4).

Deducir el origen del dataset: Analizar las columnas, sellos y nombres de variables para identificar la procedencia de los datos, entender qué universo de alimentos representan y justificar el criterio de limpieza que usaremos más adelante (1.5).

1.1 Importacion de librerias y configuracion del entorno 

In [ ]:
import pandas as pd 

1.2 Carga de los datos

Leemos el archivo dataset.csv guardado en la carpeta local data/ usando la función read_csv() de Pandas. Al usar una ruta relativa nos aseguramos de que el proyecto se pueda abrir y correr en cualquier computadora sin problemas de rutas fijas.

In [5]:
data_path = "data/dataset.csv"
df_food = pd.read_csv(data_path)

1.3 Primer vistazo a las observaciones

Mostramos las primeras filas del dataset con head() para comprobar que el archivo se haya leído bien, que las columnas se hayan separado de forma correcta y para familiarizarnos a simple vista con los datos reales que tenemos en la tabla.

In [ ]:
df_food.head(10)

1.4 Dimensiones y tipos de datos de las columnas

Revisamos el tamaño general de la tabla para saber cuántos productos y cuántas variables tenemos disponibles, y usamos info() para ver qué tipo de dato le asignó Pandas a cada columna y detectar a tiempo si hay campos con formatos incorrectos.

In [ ]:
# Dimensiones generales
print(f"Dimensiones del dataset: {df_food.shape[0]} filas y {df_food.shape[1]} columnas\n")

# Estructura técnica y tipos de datos
df_food.info()


Tambien podemos agregar al analisis si existen filas duplicadas, es decir, filas que contengan exactamente los mismos valores.

In [ ]:
duplicados = df_food.duplicated().sum()
print(f"Cantidad de filas exactamente duplicadas: {duplicados}")

Al revisar la salida de info(), notamos algunos puntos clave sobre el estado de las columnas:   

Inconsistencias en tipos de datos inferidos:

Enteros guardados como float: Aunque la variable nova_group representa niveles enteros de procesamiento (1, 2, 3 y 4), Pandas le asignó float64. Esto ocurre técnicamente por la presencia de valores nulos (NaN), ya que en el estándar tradicional de Pandas el valor nulo se representa como un número de punto flotante, forzando a toda la columna a convertirse a decimal para poder alojarlo.

Números guardados como texto: Columnas nutricionales que deberían ser valores numéricos continuos (energy_kcal, fat_100g, saturated_fat_100g, carbs_100g, sugars_100g, fiber_100g, proteins_100g, salt_100g, sodium_100g) fueron leídas como texto (str). Esto nos avisa que el archivo original trae comas, símbolos o caracteres extraños que no dejaron a Pandas interpretarlos como números reales, por lo que vamos a tener que limpiarlos y convertirlos en la Sección 2. 

Datos incompletos: El conjunto cuenta con 4.997 filas en total. Variables como allergens (3.332 datos presentes) o fiber_100g (3.495 datos presentes) tienen más de 1.500 campos vacíos cada una, por lo que va a ser necesario definir qué criterio usar con esos nulos.

Indicadores de alérgenos completos: Las columnas booleanas de alérgenos (contains_gluten, contains_fish, etc.) tienen sus 4.997 registros completos con valores de verdadero o falso (bool), listas para usarse sin necesidad de emparchar datos faltantes.

Filas duplicadas : Existen en el dataset 38 filas que contienen exactamente los mismos valores. 

1.5 Análisis del Posible Origen y Contexto de los Datos

Como para este trabajo se nos entrega el archivo dataset.csv sin una descripción ni documentación que explique de dónde salió, tenemos que analizar los datos por nuestra cuenta para deducir su origen. Esto es clave no solo para saber la fuente, sino para entender con qué tipo de alimentos estamos trabajando realmente, hasta dónde podemos extender nuestras conclusiones y evitar suposiciones equivocadas al plantear las hipótesis. Además, entender cómo se armó el archivo nos ayuda a explicar por qué encontramos columnas con números guardados como texto, errores al escribir o campos vacíos, dándonos un criterio claro para corregir estos problemas en la etapa de limpieza y análisis posterior.

Al revisar las columnas y los valores de la tabla, encontramos varias pistas claras que delatan de dónde salieron los datos:

Sistemas de sellos combinados: El dataset incluye al mismo tiempo nutriscore_grade, nova_group y ecoscore_grade. Esta combinación de clasificaciones nutricionales y ambientales en una misma base es la marca registrada de Open Food Facts, el proyecto colaborativo abierto más grande del mundo sobre alimentos.

Prefijos de idioma en las categorías: En columnas como categories los valores empiezan con etiquetas en inglés como en: (por ejemplo, en:beverages o en:snacks). Esta forma de ordenar las categorías es el estándar que usa Open Food Facts para que personas de distintos países puedan cargar productos bajo un mismo criterio.

Valores calculados cada 100 gramos: Todas las columnas de nutrientes (energy_kcal, sugars_100g, fat_100g, etc.) están medidas sobre una base de 100 g. Esto responde a las leyes de etiquetado obligatorio de la Unión Europea y otros países, donde los fabricantes deben informar los datos por cada 100 gramos o mililitros de producto.

Separación de alérgenos: La presencia de una columna con texto libre (allergens) junto a varias columnas con valores de verdadero o falso (contains_gluten, contains_fish, etc.) muestra que hubo un procesamiento previo para facilitar el análisis, separando los alérgenos más comunes de las declaraciones que vienen en los paquetes.

Con estas evidencias podemos afirmar que el archivo proviene de Open Food Facts. Esto significa que nuestra muestra representa alimentos procesados y envasados con código de barras (los que se compran en un supermercado), y explica por qué aparecen números escritos como texto o errores de tipeo, ya que muchos datos se cargan de forma comunitaria o mediante fotos tomadas con el celular.

1.6 Caracterización y Clasificación del Conjunto de Variables

Para entender la estructura de la información disponible, relevamos el significado de cada columna y clasificamos las variables según su naturaleza estadística (cualitativas o cuantitativas) y su subtipo (nominales, ordinales, continuas o discretas).

Datos descriptivos e identificación del producto:
    `product_name`: Nombre comercial del alimento. Cualitativa nominal.
    `brands`: Marca del fabricante. Cualitativa nominal.
    `categories`: Categorías taxonómicas a las que pertenece el producto Cualitativa nominal.
    `ingredients`: Listado de ingredientes declarados. Cualitativa nominal.
    `food_type`: Categoría general del producto (por ejemplo, sólido o bebida). Cualitativa nominal. Es una constante ya que todos los valores son iguales para las 4997 filas del dataset.

Sellos y sistemas de clasificación:
    `nutriscore_grade`: Evaluación global de calidad nutricional (letras A a E). Cualitativa ordinal (existe un orden de calidad evidente: la A es mejor perfil que la E).
    `nova_group`: Clasificación según grado de procesamiento industrial (niveles 1 a 4). Cualitativa ordinal (el número indica niveles crecientes de procesamiento, no una cantidad medida físicamente).
    `ecoscore_grade`: Calificación del impacto ambiental del alimento (letras A a E). Cualitativa ordinal.

Composición nutricional (medida por cada 100 g o 100 ml):
    `energy_kcal`: Aporte energético medido en kilocalorías. Cuantitativa continua.
    `fat_100g`: Contenido total de grasas en gramos. Cuantitativa continua.
    `saturated_fat_100g`: Grasas saturadas en gramos. Cuantitativa continua.
    `carbs_100g`: Hidratos de carbono en gramos. Cuantitativa continua.
    `sugars_100g`: Azúcares simples en gramos. Cuantitativa continua.
    `fiber_100g`: Fibra alimentaria en gramos. Cuantitativa continua.
    `proteins_100g`: Proteínas en gramos. Cuantitativa continua.
    `salt_100g`: Sal total en gramos. Cuantitativa continua.
    `sodium_100g`: Sodio en gramos. Cuantitativa continua.

Indicadores de alérgenos:
    `allergens`: Texto libre con los alérgenos declarados en el envase. Cualitativa nominal.
    `contains_gluten`, `contains_dairy`, `contains_eggs`, `contains_nuts`, `contains_soy`, `contains_fish`: Indicadores binarios de presencia o ausencia. Cualitativas nominales (dicotómicas / booleanas).

1.7 Diagnóstico de Valores Faltantes: La Trampa de los Alérgenos, Nulos Explícitos y Datos No Informativos

Para evaluar la calidad real de la información, no alcanza con medir únicamente los vacíos formales (`NaN`) que reconoce Pandas; es necesario analizar si existen valores que esconden la falta de dato y entender cómo se generaron las columnas derivadas.

La trampa de las columnas booleanas de alérgenos:

    Al cruzar los datos observamos que en las 1.665 filas donde el texto de `allergens` es nulo (`NaN`), todas las columnas de presencia de alérgenos (`contains_gluten`, `contains_dairy`, etc.) figuran como `False`.
    Esto demuestra que estas variables son el resultado de un procesamiento automático que buscó palabras clave sobre la columna de texto y, ante la falta de información, completó con `False` por defecto.
    Criterio metodológico: La ausencia de declaración no garantiza la ausencia del alérgeno. Un valor `False` en un producto sin datos en `allergens` representa desconocimiento, no una certificación de que el alimento sea seguro para personas con alergias. En contraste, existen 20 registros donde se declaró explícitamente `en:none`, donde sí hubo una manifestación formal de no contener alérgenos.

Valores no informativos en sellos de calidad (Nulos camuflados):

    Si bien las columnas de clasificación muestran muy pocos nulos formales, contienen categorías que representan falta de evaluación:
    `nutriscore_grade`: A los 14 `NaN` se suman 401 registros `'UNKNOWN'` y 135 `'NOT-APPLICABLE'`. Los datos no utilizables ascienden en realidad a 550 observaciones (11,0% de la muestra).
    `ecoscore_grade`: A los 13 `NaN` se suman 777 `'UNKNOWN'` y 259 `'NOT-APPLICABLE'`, alcanzando 1.049 registros sin información real (21,0% de la muestra).
    `nova_group`: Presenta 475 nulos formales (`NaN`), lo que representa un 9,5% de datos faltantes sobre el nivel de procesamiento industrial.

In [ ]:
#Verificación del patrón de alérgenos
filas_allergens_nulas = df_food[df_food['allergens'].isnull()]

columnas_alergenos = [
    'contains_gluten',
    'contains_dairy',
    'contains_eggs',
    'contains_nuts',
    'contains_soy',
    'contains_fish'
]

# Muestra cuántos True hay en cada alérgeno cuando allergens es NaN

conteo_por_alergeno = filas_allergens_nulas[columnas_alergenos].sum()
total_trues = conteo_por_alergeno.sum()

print(f"Total de registros con 'allergens' nulo: {len(filas_allergens_nulas)}")
print(f"Cantidad de valores True en columnas de alérgenos cuando 'allergens' es NaN: {total_trues}")

Conclusión del análisis de alérgenos
Hicimos esta prueba para entender cómo se construyeron las columnas booleanas (`contains_gluten`, `contains_dairy`, etc.) y no cometer el error de usarlas a ciegas.

Si miramos la tabla por arriba, parece que estas columnas están completas y listas para usar porque no tienen ningún valor nulo. Sin embargo, quisimos comprobar qué pasaba en las 1.665 filas donde el texto de allergens vino vacío (NaN).Como en Python un True equivale a 1 y un False equivale a 0, filtramos únicamente esos 1.665 productos y aplicamos una suma sobre las 6 columnas de alérgenos (contains_gluten, contains_dairy, etc.). Si al menos un producto hubiera tenido un True, la suma de esa columna tendría que dar 1 o más. Como el resultado fue estrictamente 0 en cada una, comprobamos que las 9.990 celdas analizadas (1.665 filas, 6 columnas) fueron completadas con False.

Con esto comprobamos que quien armó la base pasó un script que buscaba palabras clave en el texto de `allergens`, y como en esos 1.665 productos no había nada escrito, le asignó `False` por defecto a todo. 

Esto es un problema metodológico importante: que un producto no tenga cargada la lista de alérgenos no significa que sea apto para celíacos o libre de alérgenos, sino que simplemente no tenemos el dato. Tomar esos `False` como alimentos seguros distorsionaría cualquier análisis posterior, porque estamos mezclando productos que verdaderamente no tienen alérgenos con productos donde el usuario de la app no cargó la información.

Metodología para la auditoría de valores faltantes

Para realizar un relevamiento exhaustivo de la calidad de los datos, no es suficiente con cuantificar los valores nulos formales (`NaN`) mediante `isnull()`. En datasets reales es habitual encontrar nulos encubiertos o camuflados, es decir, registros con valores válidos a nivel de tipo de dato pero carentes de significado analítico.

La estrategia para detectar estos valores varía según la naturaleza de cada variable:

Variables numéricas: Además de los `NaN` tradicionales, estas columnas pueden ocultar valores centinela o códigos de omisión (como `-1`, `0` en contextos biológicamente imposibles o `-999`). Por las leyes físicas y químicas de los alimentos, magnitudes como gramos de nutrientes o kilocalorías no pueden ser negativas. Por lo tanto, la auditoría en estas variables se realiza inspeccionando sus rangos válidos (mínimos y máximos mediante `.describe()`) para descartar la presencia de valores fuera de dominio que actúen como nulos disfrazados.

Variables de texto (tipo `object`): No resulta viable aplicar un `value_counts()` indiscriminado sobre todas las columnas para inspeccionar sus categorías. En campos de texto abierto o descripciones con alta variabilidad (como nombres de productos o listas de ingredientes), la enorme cantidad de valores únicos saturaría el análisis e impediría una revisión manual. Por este motivo, para las variables categóricas el primer paso consiste en calcular su cardinalidad (cantidad de valores únicos). Aquellas columnas con un conjunto reducido de categorías (baja cardinalidad, como escalas nutricionales y sellos ambientales) se auditarán directamente con tablas de frecuencia completa (`value_counts(dropna=False)`). Esto permitirá contrastar las categorías observadas contra los dominios teóricos de cada escala y descubrir de forma empírica qué etiquetas (como `"unknown"` o `"not-applicable"`) actúan como valores faltante.

Por último, para las variables de texto libre o con alta cardinalidad (tales como nombres de productos, marcas o listas de ingredientes), donde la dispersión de valores únicos descarta tanto la revisión manual mediante tablas de frecuencias como el cálculo de estadísticos descriptivos, la detección de nulos encubiertos se aborda mediante técnicas de filtrado heurístico y análisis de patrones. Este procedimiento consiste en auditar la existencia de cadenas no informativas a través de dos mecanismos: por un lado, la identificación de registros vacíos o compuestos exclusivamente por espacios en blanco que el sistema no tipifica nativamente como ausentes; por otro, la aplicación de expresiones regulares para rastrear de forma automatizada patrones y términos genéricos de omisión comúnmente introducidos por usuarios o interfaces (tales como "none", "unknown", "n/a", guiones o signos de interrogación). De esta manera, es posible cuantificar e incorporar al diagnóstico de calidad aquellos registros sintácticamente válidos pero carentes de aporte semántico sin requerir una inspección individualizada de cada entrada.

In [ ]:
for col in df_food.columns:
    # Verificamos si la columna es de texto
    if df_food[col].dtype == 'str':
        cantidad_unicos = df_food[col].nunique(dropna=False)
        print("Columna:", col, "-> Valores únicos:", cantidad_unicos)

In [ ]:
cols_nutrientes = [
    'energy_kcal', 'fat_100g', 'saturated_fat_100g', 'carbs_100g', 
    'sugars_100g', 'fiber_100g', 'proteins_100g', 'salt_100g', 'sodium_100g'
]
# Convertimos temporalmente a número para evaluar mínimos matemáticos reales
df_food[cols_nutrientes].apply(pd.to_numeric, errors='coerce').agg(["min","max"])


Al auditar los valores máximos de las variables cuantitativas nutricionales, se constata la presencia de severas inconsistencias de carga que violan los límites físicos, biológicos y estequiométricos de los alimentos. Dado que las magnitudes están estandarizadas cada 100 gramos de producto, ningún nutriente individual puede superar matemáticamente los 100 g ni la energía sobrepasar las 900 kcal correspondientes a la grasa pura, umbrales que resultan ampliamente transgredidos al observar registros anómalos de cientos de gramos en macronutrientes, más de 5.200 kcal de energía y un valor espurio de orden billonario en fibra. Metodológicamente, se concluye que estas cifras desproporcionadas no corresponden a códigos centinela utilizados para denotar valores faltantes, sino a registros corruptos y errores de carga en la recolección de datos; esta evidencia justifica empíricamente por qué resulta inviable calcular estadísticos descriptivos como medias o desvíos estándar en esta etapa, debiendo documentar el diagnóstico en este primer ejercicio para luego sanear, filtrar o imputar dichas aberraciones durante la fase de curación y limpieza.

In [ ]:
# Lista de variables cualitativas y ordinales de baja cardinalidad
cols_baja_cardinalidad = ['nutriscore_grade', 'ecoscore_grade', 'nova_group']

# Inspección exhaustiva de categorías y detección de nulos encubiertos
for col in cols_baja_cardinalidad:
    print(f"Distribución de valores en: {col}")
    print(df_food[col].value_counts(dropna=False))

Al realizar las tablas de frecuencias sobre las variables cualitativas y ordinales de baja cardinalidad, se evidenció con claridad la discrepancia entre los nulos explícitos y los reales. En nutriscore_grade y ecoscore_grade, las ausencias formales mediante NaN apenas alcanzan 14 y 13 registros respectivamente; no obstante, al contrastar los valores observados contra los estándares oficiales se descubrió la presencia de etiquetas de omisión del sistema como "UNKNOWN" y "NOT-APPLICABLE", las cuales acumulan 536 y 1.036 casos encubiertos en cada escala.En la variable ecoscore_grade tambien podemos mecionar que en la tabla de frecuencias aparecen 257 casos de 'A-PLUS' y 107 de 'F'. Como la escala estándar oficial va de A a E, podemos afirmar que el dataset incluye categorías atípicas o fuera de la escala tradicional. Por su parte, la variable ordinal nova_group presentó únicamente valores dentro del dominio admisible (grupos del 1 al 4) y ausencias explícitas (NaN), descartando la utilización de códigos numéricos de relleno. Este contraste empírico confirma la necesidad metodológica de la auditoría de dominio, documentando de forma fehaciente que los verdaderos faltantes en las escalas ambientales y nutricionales superan ampliamente a los reportados por métodos tradicionales como isnull().

In [ ]:
# Definimos los patrones de texto que podrian representar valores nulos no explicitos. Estos son valores que, aunque no sean NaN, indican que la información no está disponible o no es aplicable.
valores_no_informativos = ["not-applicable", "unknown", "none", "n/a", "null", "none", "nan", "void", "empty","-", "?"]

resumen_nulos_completo = [] 
for col in df_food.columns:
    # Nulos tradicionales (NaN)
    nulos_nan = df_food[col].isnull().sum()
    
    # Nulos camuflados en texto (ignorando nulos previos y espacios extra)
    if (df_food[col].dtype == 'string'):
        nulos_texto = df_food[col].dropna().astype(str).str.strip().str.lower().isin(valores_no_informativos).sum()
    else:
        nulos_texto = 0
        
    total_faltantes = nulos_nan + nulos_texto
    porcentaje_faltantes = round(total_faltantes/ len(df_food)* 100, 2)

    resumen_nulos_completo.append({
        "columna" : col,
        "nulos_explicitos" : nulos_nan,
        "nulos_camuflados" : nulos_texto,
        "total_faltantes" : total_faltantes,
        "porcentaje_faltantes" : porcentaje_faltantes
    })
df_datos_faltantes = pd.DataFrame(resumen_nulos_completo).sort_values(by="porcentaje_faltantes", ascending=False)
df_datos_faltantes   



Para determinar qué palabras pueden actuar como nulos camuflados sin tener que mirar todo el archivo de antemano, nos basamos en cómo funcionan los sistemas y los formularios de carga de datos en el mundo real. Cuando un software exporta información a un archivo plano como un CSV, la falta de datos suele escribirse automáticamente con palabras técnicas reservadas (como "null", "none" o "nan"). Al mismo tiempo, en cualquier aplicación o página web donde una persona o un algoritmo clasifica un producto y no tiene el dato certero, la interfaz suele ofrecer opciones predeterminadas para no dejar la celda vacía, tales como "unknown", "not-applicable", "n/a" o simplemente un guion. Por lo tanto, no se eligen estas palabras al azar ni por haber visto el dataset completo, sino que se define una lista de búsqueda estándar a partir de estas convenciones habituales de carga y exportación de datos, permitiendo auditar de manera objetiva y reproducible si el dataset contiene este tipo de respuestas por defecto.

Seccion 1.8 : Comportamiento y distribucion de las variables y outliers

Después de identificar y contar los valores que faltan, analizamos qué nos muestran los datos que sí son válidos en el dataset. En las variables de texto y categorías con pocas opciones, los porcentajes nos dejan ver claramente cómo está compuesta la muestra: más del 58% de los productos corresponden al Grupo 4 de NOVA (alimentos ultraprocesados), lo que demuestra que el dataset está muy sesgado hacia productos envasados e industriales; en Nutri-Score la mayoría se concentra en las notas C (24,2%) y E (22,5%), reflejando una calidad nutricional baja; en Eco-Score aparecen etiquetas raras o viejas como A-PLUS y F; y en los alérgenos lo que más abunda es gluten (32%) y lácteos (30%). Por el lado de los nutrientes cuantitativos (grasas, azúcares, etc.), no calculamos promedios, porcentajes ni gráficos porque todavía figuran como texto (object) en Pandas y además tienen errores de carga, como valores que superan por mucho los 100 gramos por porción. Intentar medir su distribución ahora nos daría números sin ningún sentido real, así que dejamos su conversión a números y su análisis para la Sección 2, una vez que limpiemos esos registros erróneos.
Respecto a los valores atípicos (outliers), su análisis estadístico formal también se deja para la Sección 2. Con los datos en crudo solo pudimos notar a simple vista errores groseros que rompen los límites de la física, como cantidades mayores a 100 gramos o cifras billonarias. Pero para usar técnicas reales de detección de atípicos (como boxplots), primero es obligatorio convertir las columnas de texto a números y limpiar esas cargas corruptas, ya que de lo contrario cualquier cuenta o gráfico quedaría completamente deformado por esos valores gigantescos.


In [ ]:
# Distribución y prevalencia de alérgenos en el dataset (porcentaje sobre el total)
columnas_alergenos = [
        'contains_gluten', 'contains_dairy', 'contains_eggs', 
        'contains_nuts', 'contains_soy', 'contains_fish'
    ]
round(df_food[columnas_alergenos].mean() * 100, 2)


Para medir la presencia de cada alérgeno en el dataset, calculamos el promedio de las seis columnas booleanas con .mean() * 100. Como en Python los valores True valen 1 y los False valen 0, Pandas divide la cantidad de positivos por el total de filas, dándonos directamente el porcentaje de productos que contiene cada alérgeno. Los resultados muestran una presencia claramente liderada por ingredientes típicos de alimentos envasados: el gluten (32,00%) y los lácteos (29,86%) aparecen en casi un tercio de la muestra, seguidos por la soja (16,95%) y los frutos secos (11,81%), mientras que el huevo (6,36%) y el pescado (2,24%) tienen una frecuencia mucho más baja y residual dentro de los productos analizados.